# Variable Selection as Screening

Stepwise and best-subset procedures are common screening tools in regression software. This notebook implements the same ideas in Python and emphasizes their limitations.

By the end of this notebook, you should be able to:

- run forward/backward stepwise screening with transparent Python code;
- enumerate best subsets and compare adjusted $R^2$, AIC, BIC, Mallows $C_p$, and residual standard error;
- explain why selection methods induce Type I and Type II error risks;
- enforce hierarchy and validate the chosen model after screening.

In [ ]:
from lite_setup import ensure_packages
await ensure_packages()

In [ ]:
from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from checks import check_nonempty

sales = pd.read_csv(Path('data/territory_sales.csv'))
response = 'Sales'
predictors = ['Time', 'MarketPotential', 'Advertising', 'MarketShare', 'ChangeAccounts', 'Workload', 'Rating', 'CompetitorPressure']
sales[predictors + [response]].head()

## Stepwise Regression

Stepwise regression is an iterative screening procedure. Forward selection starts with no predictors and adds candidates. Backward elimination starts with the full model and removes weak candidates. Mixed stepwise can add and remove as the model changes.

The lecture warning matters: repeated tests inflate error rates. Treat stepwise results as an initial model-building suggestion, not as final evidence.

In [ ]:
def fit_model(data, response, terms):
    formula = response + ' ~ ' + (' + '.join(terms) if terms else '1')
    return smf.ols(formula, data=data).fit()

def stepwise_pvalue(data, response, candidates, enter=0.05, remove=0.10):
    selected = []
    history = []
    remaining = list(candidates)
    while remaining:
        trial = []
        for term in remaining:
            model = fit_model(data, response, selected + [term])
            trial.append((model.pvalues.get(term, np.nan), term, model.aic))
        trial = sorted(trial)
        best_p, best_term, best_aic = trial[0]
        if best_p >= enter:
            break
        selected.append(best_term)
        remaining.remove(best_term)
        history.append({'action': 'add', 'term': best_term, 'p_value': best_p, 'aic': best_aic})
        changed = True
        while changed and selected:
            changed = False
            model = fit_model(data, response, selected)
            worst_term = model.pvalues.drop('Intercept').idxmax()
            worst_p = model.pvalues[worst_term]
            if worst_p > remove:
                selected.remove(worst_term)
                remaining.append(worst_term)
                history.append({'action': 'remove', 'term': worst_term, 'p_value': worst_p, 'aic': model.aic})
                changed = True
    return selected, pd.DataFrame(history)

selected, history = stepwise_pvalue(sales, response, predictors)
print(check_nonempty(selected, label='stepwise selection'))
print('Selected terms:', selected)
history

In [ ]:
step_model = fit_model(sales, response, selected)
print(step_model.summary().tables[1])
print('Adjusted R-squared:', round(step_model.rsquared_adj, 4))
print('AIC:', round(step_model.aic, 2))

## Best Subset Regression

Best subset regression fits every candidate subset and then compares the best models of each size. Common criteria:

- $R^2$: increases when predictors are added, so it is not enough by itself.
- Adjusted $R^2$: penalizes extra predictors.
- $s = \sqrt{MSE}$: residual standard error, smaller is better.
- AIC/BIC: information criteria, smaller is better.
- Mallows $C_p$: one version is

$$C_p = \frac{SSE_p}{MSE_{full}} + 2(p+1) - n,$$

where $p$ is the number of predictors in the subset and $p+1$ counts the intercept.

In [ ]:
def best_subsets(data, response, candidates):
    full = fit_model(data, response, candidates)
    mse_full = full.mse_resid
    rows = []
    for r in range(1, len(candidates) + 1):
        for combo in itertools.combinations(candidates, r):
            model = fit_model(data, response, list(combo))
            p = len(combo)
            cp = model.ssr / mse_full + 2 * (p + 1) - model.nobs
            rows.append({'predictors': combo, 'num_predictors': p, 'adj_r2': model.rsquared_adj,
                         'r2': model.rsquared, 's': np.sqrt(model.mse_resid), 'aic': model.aic,
                         'bic': model.bic, 'cp': cp})
    return pd.DataFrame(rows)

subsets = best_subsets(sales, response, predictors)
best_by_size = subsets.sort_values('adj_r2', ascending=False).groupby('num_predictors').head(1).sort_values('num_predictors')
best_by_size

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].plot(best_by_size['num_predictors'], best_by_size['adj_r2'], marker='o')
axes[0].set_xlabel('Number of predictors')
axes[0].set_ylabel('Best adjusted R-squared')
axes[0].set_title('Adjusted R-squared')
axes[1].plot(best_by_size['num_predictors'], best_by_size['cp'], marker='o')
axes[1].plot(best_by_size['num_predictors'], best_by_size['num_predictors'] + 1, linestyle='--', color='gray', label='p + 1')
axes[1].set_xlabel('Number of predictors')
axes[1].set_ylabel('Mallows Cp')
axes[1].set_title('Cp')
axes[1].legend()
axes[2].plot(best_by_size['num_predictors'], best_by_size['bic'], marker='o')
axes[2].set_xlabel('Number of predictors')
axes[2].set_ylabel('BIC')
axes[2].set_title('BIC')
plt.tight_layout()

Interpretation: look for a small model after which improvement is minor. If the selected model contains an interaction or polynomial term, preserve hierarchy by keeping the corresponding lower-order terms unless there is a strong course-approved reason not to.

In [ ]:
bic_row = subsets.sort_values('bic').iloc[0]
bic_terms = list(bic_row['predictors'])
bic_model = fit_model(sales, response, bic_terms)
pd.DataFrame([
    {'model': 'stepwise', 'terms': tuple(selected), 'adj_r2': step_model.rsquared_adj, 'aic': step_model.aic, 'bic': step_model.bic},
    {'model': 'best_bic', 'terms': tuple(bic_terms), 'adj_r2': bic_model.rsquared_adj, 'aic': bic_model.aic, 'bic': bic_model.bic},
])

## Caveats

Selection procedures perform many comparisons, can miss the true structure, can select models that violate hierarchy, and do not check assumptions. After screening, always run diagnostics, assess subject-matter plausibility, and validate prediction performance when possible.